# 02 · Limpieza, transformación e integración (ETL)

Pasos:
1. Convertir las fechas y crear el código de distrito.
2. Quedarse con el periodo enero 2015 – septiembre 2026.
3. Revisar duplicados, nulos y valores negativos.
4. Calcular el indicador de separación.
5. Unir con población y mapa, y guardar los resultados en `data/processed/`.

In [1]:
from pathlib import Path
import pandas as pd

# Carpeta del proyecto (funciona si el notebook se abre desde la raíz o desde notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
OUT = ROOT / 'data' / 'processed'
FIG = ROOT / 'docs' / 'figures'
import geopandas as gpd

INICIO, FIN = '2015-01-01', '2026-09-01'

## 1. Fechas y código de distrito
La columna `MONTH` viene como texto (`2026 / 10`). Se convierte a fecha para poder ordenar y filtrar.
El código `boro_cd` une el borough y el número de distrito (por ejemplo, Brooklyn 6 → 306). Es la misma clave que usan el mapa y la tabla de población.

In [2]:
bruto = pd.read_csv(RAW / 'tonnage.csv')
print('Datos en bruto:', bruto.shape)

df = bruto.copy()
df['month'] = pd.to_datetime(df['MONTH'].str.replace(' ', ''), format='%Y/%m')
df['boro_cd'] = df['BOROUGH_ID'] * 100 + df['COMMUNITYDISTRICT']
df[['MONTH', 'month', 'BOROUGH', 'COMMUNITYDISTRICT', 'boro_cd']].head()

Datos en bruto: (25355, 12)


,MONTH,month,BOROUGH,COMMUNITYDISTRICT,boro_cd
0,2026 / 10,2026-10-01,Bronx,1,201
1,2026 / 10,2026-10-01,Bronx,2,202
2,2026 / 10,2026-10-01,Bronx,3,203
3,2026 / 10,2026-10-01,Bronx,4,204
4,2026 / 10,2026-10-01,Bronx,5,205


## 2. Periodo de estudio
Se elige 2015-2026 porque es un periodo reciente y, como se verá en el EDA, los vacíos de papel y MGP están en años anteriores (1990-2004). Septiembre de 2026 es el último mes completo al iniciar el proyecto.

In [3]:
df = df[df['month'].between(INICIO, FIN)]

# Solo se usan las tres fracciones del estudio; los orgánicos quedan fuera del alcance
df = df.rename(columns={'REFUSETONSCOLLECTED': 'basura',
                        'PAPERTONSCOLLECTED': 'papel',
                        'MGPTONSCOLLECTED': 'mgp'})
df = df[['month', 'boro_cd', 'BOROUGH', 'basura', 'papel', 'mgp']]
print('Registros:', len(df), '| Distritos:', df['boro_cd'].nunique(), '| Meses:', df['month'].nunique())

Registros: 8319 | Distritos: 59 | Meses: 141


## 3. Control de calidad
Se revisan los problemas más comunes. Si apareciera alguno, habría que decidir qué hacer antes de seguir.

In [4]:
fracciones = ['basura', 'papel', 'mgp']
print('Duplicados exactos:      ', df.duplicated().sum())
print('Distrito-mes repetidos:  ', df.duplicated(['boro_cd', 'month']).sum())
print('Valores nulos:           ', df[fracciones].isna().sum().sum())
print('Valores negativos:       ', (df[fracciones] < 0).sum().sum())
print('Combinaciones esperadas: ', 59 * 141, '| encontradas:', len(df))

Duplicados exactos:       0
Distrito-mes repetidos:   0
Valores nulos:            0
Valores negativos:        0
Combinaciones esperadas:  8319 | encontradas: 8319


No hay duplicados, nulos, negativos ni meses faltantes: los 59 distritos tienen sus 141 meses. Por eso **no fue necesario eliminar ni imputar registros**.

## 4. Indicador de separación
$$\text{separación (\%)} = \frac{\text{papel} + \text{MGP}}{\text{basura} + \text{papel} + \text{MGP}} \times 100$$

Indica qué parte de lo recolectado salió como reciclable. Al comparar periodos largos se suman primero las toneladas y luego se divide; no se promedian porcentajes.

In [5]:
df['total'] = df[fracciones].sum(axis=1)
df['separacion_pct'] = 100 * (df['papel'] + df['mgp']) / df['total']
df['separacion_pct'].describe().round(1)

count    8319.0
mean       17.4
std         5.4
min         3.2
25%        13.6
50%        17.1
75%        20.8
max        35.4
Name: separacion_pct, dtype: float64

## 5. Integración con población y mapa
La población de 2010 (último censo de la fuente) solo se usa como contexto y para los nombres de los distritos.

In [6]:
BORO_ID = {'Manhattan': 1, 'Bronx': 2, 'Brooklyn': 3, 'Queens': 4, 'Staten Island': 5}
pob = pd.read_csv(RAW / 'population.csv')
pob['boro_cd'] = pob['Borough'].map(BORO_ID) * 100 + pob['CD Number']
pob = pob.rename(columns={'CD Name': 'nombre', '2010 Population': 'poblacion_2010'})

df = df.merge(pob[['boro_cd', 'nombre', 'poblacion_2010']], on='boro_cd', how='left')
print('Distritos sin nombre tras la unión:', df['nombre'].isna().sum())
df.head()

Distritos sin nombre tras la unión: 0


,month,boro_cd,BOROUGH,basura,papel,mgp,total,separacion_pct,nombre,poblacion_2010
0,2026-09-01,201,Bronx,3517.0,261.4,140.8,3919.2,10.262298,"Melrose, Mott Haven, Port Morris",91497
1,2026-09-01,202,Bronx,8050.7,279.3,214.3,8544.3,5.776951,"Hunts Point, Longwood",52246
2,2026-09-01,203,Bronx,2323.6,196.4,167.5,2687.5,13.540465,"Morrisania, Crotona Park East",79762
3,2026-09-01,204,Bronx,4372.1,259.8,290.1,4922.0,11.172288,"Highbridge, Concourse Village",146441
4,2026-09-01,205,Bronx,3704.5,268.7,319.6,4292.8,13.704342,"University Hts., Fordham, Mt. Hope",128200


Para el mapa se usa el último mes completo (septiembre 2026). Los 12 polígonos sin datos quedan vacíos y se pintarán en gris.

In [7]:
mapa = gpd.read_file(RAW / 'districts.geojson')
mapa['boro_cd'] = mapa['boro_cd'].astype(int)
ultimo = df[df['month'] == FIN]
mapa = mapa.merge(ultimo.drop(columns='month'), on='boro_cd', how='left')
print('Polígonos con dato:', mapa['separacion_pct'].notna().sum(), 'de', len(mapa))

Polígonos con dato: 59 de 71


## Carga
Se guardan los archivos que usarán QGIS, Power BI y los siguientes notebooks.

In [8]:
OUT.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT / 'residuos_mensuales.csv', index=False)
pob.to_csv(OUT / 'poblacion_historica.csv', index=False)
mapa.to_file(OUT / 'distritos_indicadores.geojson', driver='GeoJSON')

resumen = {'filas_brutas': len(bruto), 'filas_procesadas': len(df),
           'distritos': df['boro_cd'].nunique(), 'meses': df['month'].nunique(),
           'duplicados': 0, 'nulos': 0, 'poligonos_sin_dato': int(mapa['separacion_pct'].isna().sum())}
pd.Series(resumen)

filas_brutas          25355
filas_procesadas       8319
distritos                59
meses                   141
duplicados                0
nulos                     0
poligonos_sin_dato       12
dtype: int64